## Libraries

In [1]:
import pandas as pd
import numpy as np

# Valid hours

The first assumption is that you are not supposed to be cycling before sunrise or after sunset 

In [11]:
hourly_2020_21 = pd.read_csv(r"raw_data\hourly_weather_data_2020_21.csv")
hourly_2022_23 = pd.read_csv(r"raw_data\hourly_weather_data_2022_23.csv")
hourly_2024_25 = pd.read_csv(r"raw_data\hourly_weather_data_2024_25.csv")
hourly = pd.concat([hourly_2020_21, hourly_2022_23, hourly_2024_25])
hourly["time"] = pd.to_datetime(hourly["time"])
hourly["date"] = pd.to_datetime(hourly["time"].dt.date)
# hourly.head(2)

In [12]:
daily_2020_21 = pd.read_csv(r"raw_data\daily_weather_data_2020_21.csv")
daily_2022_23 = pd.read_csv(r"raw_data\daily_weather_data_2022_23.csv")
daily_2024_25 = pd.read_csv(r"raw_data\daily_weather_data_2024_25.csv")
daily = pd.concat([daily_2020_21, daily_2022_23, daily_2024_25])
daily["time"] = pd.to_datetime(daily["time"])
daily["sunrise"] = pd.to_datetime(daily["sunrise"])
daily["sunset"] = pd.to_datetime(daily["sunset"])
daily = daily[["time", "name", "sunrise", "sunset"]]
daily.rename(columns={"time" : "date"}, inplace=True)
# daily.head(2)

In [13]:
valid_hours = hourly.merge(daily, on=["date", "name"], how="left")

diff_sunrise = valid_hours["time"] - valid_hours["sunrise"]
diff_sunset = valid_hours["sunset"] - valid_hours["time"]
tolerance = 20 # tolerance (in minutes) before and after sunrise/sunset

mask = ((diff_sunrise >= pd.Timedelta(0)) | (diff_sunrise.abs() <= pd.Timedelta(minutes= tolerance))) & ((diff_sunset >= pd.Timedelta(0)) | (diff_sunset.abs() <= pd.Timedelta(minutes= tolerance)))
valid_hours = valid_hours.loc[mask]

In [14]:
# valid_hours.loc[valid_hours["name"] == "Sondrio"]

# Intermediate scores

Within daylight_duration:
* Temperature must be inside a range
* Little rain (temperature-dependent)
* No snow
    * falling snow
    * on the ground
* Little wind
    * low constant speed
    * little gusts

In [15]:
feature_engineering = valid_hours.copy()

Step 1 (Temperature):
* Below/above a threshold the score is 0 (too cold or too hot)
* Inside a comfort range the score is 1
* The score outside the comfort  is linear (ratio between the distance from the comfort threhsold and the outside-range) 

In [16]:
T_low__acceptable_threshold = 5
T_high__acceptable_threshold = 30
T_low__comfort_threshold = 15
T_high__comfort_threshold = 25

T = feature_engineering["apparent_temperature"]

T_outside_acceptable = (T < T_low__acceptable_threshold) | (T > T_high__acceptable_threshold)
T_inside_comfort = (T >= T_low__comfort_threshold) & (T <= T_high__comfort_threshold)

T_intermediate = np.where(T < T_low__comfort_threshold,
                        1 - (T_low__comfort_threshold - T) / 
                            (T_low__comfort_threshold - T_low__acceptable_threshold),
                        1 - (T - T_high__comfort_threshold) / 
                            (T_high__acceptable_threshold - T_high__comfort_threshold)

)

feature_engineering["T_score"] = np.where(T_outside_acceptable, 0,
                                            np.where(T_inside_comfort, 1,
                                            T_intermediate)
)

Step 2 (Rain):
* Above a threshold the score is 0 (too much rain)
* When there's no rain, the score is 1
* If there is rain, a combination of rain-temperature is evaluated:
    * if temperature is above a threshold and rain below another threshold, it is acceptable (summer rain)

In [17]:
R_high_acceptable_threshold = 4
R_low_acceptable_temperature = 10

R = feature_engineering["rain"]

R_too_high = (R > R_high_acceptable_threshold)
no_rain = (R == 0)

intermediate_R_score = 1 - (R / R_high_acceptable_threshold)
intermediate_T_score = (T - R_low_acceptable_temperature) / (T_high__acceptable_threshold - R_low_acceptable_temperature)

# R_intermediate = (intermediate_R_score + intermediate_T_score)/2
R_intermediate = np.where(T <= R_low_acceptable_temperature,
                          0,
                          (intermediate_R_score * intermediate_T_score)
)

feature_engineering["R_score"] = np.where(R_too_high, 0,
                                            np.where(no_rain, 1,
                                            R_intermediate)
)
# feature_engineering.loc[feature_engineering["rain"] != 0]

Step 3 (Snow):
* Boolean value
    * No falling snow
    * No snow on the ground

In [18]:
snowfall = feature_engineering["snowfall"]
snow_on_ground = feature_engineering["snow_depth"]

no_snow = snowfall > 0
no_snow_on_the_ground = snow_on_ground > 0

feature_engineering["S_score"] = np.where(no_snow | no_snow_on_the_ground, 0,
                                            1)

Step 4 (Wind):
* the wind_speed above a threshold is dangerous, as for wind_gusts
* the final score is dictated by the worst condition (speed or gusts)

In [19]:
W_speed_threshold = 30
W_gusts_threshold = 40

S = feature_engineering["wind_speed_10m"]
G = feature_engineering["wind_gusts_10m"]

intermediate_S_score = 1 - (S / W_speed_threshold)
intermediate_G_score = 1 - (G / W_gusts_threshold)
# weight_G = 2
# intermediate_W_score = (intermediate_S_score + (weight_G * intermediate_G_score))/(1 + weight_G)
intermediate_W_score = np.minimum(intermediate_S_score, intermediate_G_score)


feature_engineering["W_score"] = np.where(intermediate_W_score >= 0,
                                        intermediate_W_score,
                                        0
)

# Overall weather score

Intermediate scores are multiplied by hour and summed by day, so that days with more daylight have higher scores

In [20]:
feature_engineering["Hourly_weather_score"] = feature_engineering["T_score"] * feature_engineering["R_score"] * feature_engineering["S_score"] * feature_engineering["W_score"]

In [21]:
feature_engineering.to_csv("processed_data/feature_engineering_extended.csv", index=False)

In [22]:
feature_engineering_grouped = feature_engineering.copy()
feature_engineering_grouped = feature_engineering_grouped.groupby(["name", "date"])["Hourly_weather_score"].sum().to_frame().reset_index()
feature_engineering_grouped.to_csv("processed_data/feature_engineering_grouped.csv", index=False)